In [5]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Literal
import operator
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langgraph.graph import StateGraph
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.chat_models import AzureChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
import uuid
# --- Load environment variables ---
load_dotenv()
memory_checkpointer = InMemorySaver()

# --- Tools ---
tavily = TavilySearchResults(k=3)
tools = [tavily]

# --- LLM ---
llm = AzureChatOpenAI(
    deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["OPENAI_API_VERSION"],
    openai_api_key=os.environ["AZURE_OPENAI_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)

# --- Prompt templates ---
planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a planner that breaks down complex tasks into simple steps."),
    MessagesPlaceholder("messages")
])

executor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant executing a task step by step using tools when needed. Here is what you remember: {memory}"),
    MessagesPlaceholder("messages"),
    MessagesPlaceholder("scratchpad"),
])

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize this memory in a concise, informative report."),
    ("human", "{memory}")
])

# --- State ---
class MemoryEntry(TypedDict):
    step: str
    content: str
    success: bool

class AgentState(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage | ToolMessage], operator.add]
    memory: List[MemoryEntry]
    plan: List[str]
    step: int

# --- Nodes ---
async def plan_task(state: AgentState) -> AgentState:
    response = await llm.ainvoke(planner_prompt.format_messages(messages=state["messages"]))
    print("\n📌 Plan generated:", response.content)
    steps = [s.strip("- ") for s in response.content.split("\n") if s.strip()]
    return {
        **state,
        "plan": steps,
        "step": 0,
        "messages": state["messages"][-1:] + [response]
    }

async def execute_step(state: AgentState) -> AgentState:
    if state["step"] >= len(state["plan"]):
        return state
    step_number = state["step"] + 1
    current_step = state["plan"][state["step"]]
    task_msg = HumanMessage(content=f"Please carry out the following step in the task plan: '{current_step}'. If needed, use any tools to assist you.")

    tool_result = await llm.ainvoke(executor_prompt.format_messages(
        messages=state["messages"] + [task_msg],
        scratchpad=[],
        memory="\n".join(f"{m['step']}: {m['content']}" for m in state["memory"][-5:])
    ))

    print(f"\n🔧 Executing Step {step_number}: {current_step}")
    print("Response:", tool_result.content)

    success = not any(keyword in tool_result.content.lower() for keyword in ["error", "not possible", "failed", "couldn't"])
    memory_entry = {"step": current_step, "content": tool_result.content, "success": success}

    return {
        **state,
        "messages": [task_msg, tool_result],
        "memory": state["memory"] + [memory_entry],
        "step": state["step"] + 1
    }


def maybe_replan(state: AgentState) -> AgentState:
    if state["memory"] and not state["memory"][-1]["success"]:
        print("\n🔁 Re-planning due to issue in execution...")
        return {**state, "step": 0, "plan": [], "messages": state["messages"]}
    return state

async def summarize_memory(memory: List[MemoryEntry]) -> str:
    combined = "\n".join(f"Step: {m['step']}\n{m['content']}" for m in memory)
    summary = await llm.ainvoke(summary_prompt.format_messages(memory=combined))
    return summary.content


def compress_memory(state: AgentState) -> AgentState:
    if len(state["memory"]) > 10:
        memory_text = "\n".join(f"{m['step']}: {m['content']}" for m in state["memory"])
        summary = llm.ainvoke(summary_prompt.format_messages(memory=memory_text))
        print("\n📝 Memory summarized.")
        return {
            **state,
            "memory": [{"step": "summary", "content": summary.content, "success": True}],
            "messages": state["messages"][-2:]
        }
    return state

# --- Graph ---
builder = StateGraph(AgentState)
builder.add_node("planner", RunnableLambda(plan_task))
builder.add_node("executor", RunnableLambda(execute_step))
builder.add_node("compressor", RunnableLambda(compress_memory))
builder.add_node("replanner", RunnableLambda(maybe_replan))

builder.set_entry_point("planner")

builder.add_conditional_edges(
    "planner",
    lambda state: "executor" if "plan" in state and len(state["plan"]) > 0 else None
)

def executor_branch(state: AgentState) -> str | None:
    if state["step"] < len(state["plan"]):
        if state["memory"] and not state["memory"][-1]["success"]:
            return "replanner"
        else:
            return "compressor"
    return None

builder.add_conditional_edges("executor", executor_branch)

builder.add_conditional_edges(
    "replanner",
    lambda state: "planner" if len(state["plan"]) == 0 else "executor"
)

graph = builder.compile()

# --- Async Run Function ---
async def run_agent():
    thread_id = "rocket_launch_research"

    # Load saved state asynchronously if available
    config = {
            "configurable": {
            "thread_id": thread_id,
            "checkpoint_ns": "default"
        }
    }
    saved_state = await memory_checkpointer.aget(config)

    initial_state = {
            "messages": [HumanMessage(content="Help me research a report on the environmental impact of rocket launches.")],
            "memory": [],
            "plan": [],
            "step": 0
        }
    if saved_state:
        initial_state.update(saved_state)

    # Run graph asynchronously with checkpointer and thread_id
    final_state = await graph.ainvoke(
        initial_state,
        config={
            "checkpointer": memory_checkpointer,
            "thread_id": thread_id,
            "checkpoint_ns": "default"
        }
    )
    # Check if 'channel_values' is in final_state, or create a minimal one
    if "channel_values" not in final_state:
        final_state["channel_values"] = {"default": {}}

    final_state["pending_sends"] = {}

    new_versions = {channel: "v1" for channel in final_state["channel_values"]}
    final_state["id"] = str(uuid.uuid4()) 
    memory_checkpointer.put(config, final_state, {}, new_versions)
    
    print("\n🧠 Memory:")
    for i, mem in enumerate(final_state["memory"], 1):
        print(f"{i}. Step: {mem['step']}\n{mem['content']}\n")

    print("\n📝 Summary:")
    print(await summarize_memory(final_state["memory"]))

    
    return final_state
# Run async agent
final_state = await run_agent()


C:\Users\rishi\AppData\Local\Temp\ipykernel_32556\4236173471.py:22: LangChainDeprecationWarning: The class `AzureChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import AzureChatOpenAI``.
  llm = AzureChatOpenAI(



📌 Plan generated: Sure! Here’s a step-by-step guide to help you research and compile a report on the environmental impact of rocket launches:

### Step 1: Define the Scope of the Report
- **Objective**: Determine what you want to cover (e.g., air quality, carbon emissions, noise pollution, impact on wildlife).
- **Audience**: Identify who will read the report (e.g., policymakers, researchers, general public).

### Step 2: Background Research
- **History of Rocket Launches**: Briefly summarize the evolution of rocket technology and frequency of launches.
- **Current Launch Statistics**: Gather data on the number of launches per year, types of rockets used, and key players in the industry.

### Step 3: Identify Key Environmental Impacts
- **Air Quality**:
  - Research emissions released during launches (e.g., black carbon, hydrogen chloride).
- **Carbon Footprint**:
  - Estimate the total carbon emissions from rocket launches and compare with other industries.
- **Noise Pollution**:
  -